In [1]:
import sys
print(sys.executable)

c:\ShadowFox-ML-Internship\Advanced\NLP-Language-Model\.venv\Scripts\python.exe


In [2]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Device:", "CUDA GPU" if torch.cuda.is_available() else "CPU")

PyTorch: 2.14.1+cpu
Transformers: 5.20.2
Device: CPU


In [3]:
from transformers import pipeline

MODEL_NAME = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model=MODEL_NAME,
    device=-1
)

print("DistilBERT sentiment model loaded successfully.")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBERT sentiment model loaded successfully.


In [4]:
sentences = [
    "I absolutely love this product.",
    "This is the worst experience I have ever had.",
    "The service was good, but the delivery was disappointing."
]

results = sentiment_analyzer(sentences)

for sentence, result in zip(sentences, results):
    print(f"Text: {sentence}")
    print(f"Predicted sentiment: {result['label']}")
    print(f"Confidence score: {result['score']:.4f}")
    print("-" * 50)

Text: I absolutely love this product.
Predicted sentiment: POSITIVE
Confidence score: 0.9999
--------------------------------------------------
Text: This is the worst experience I have ever had.
Predicted sentiment: NEGATIVE
Confidence score: 0.9998
--------------------------------------------------
Text: The service was good, but the delivery was disappointing.
Predicted sentiment: NEGATIVE
Confidence score: 0.9960
--------------------------------------------------


In [5]:

import pandas as pd

test_cases = [
    {
        "Category": "Clearly positive",
        "Text": "The food was delicious and the staff were wonderful.",
        "Expected": "POSITIVE"
    },
    {
        "Category": "Clearly negative",
        "Text": "The product broke after one day and customer support was terrible.",
        "Expected": "NEGATIVE"
    },
    {
        "Category": "Negation",
        "Text": "This movie was not good.",
        "Expected": "NEGATIVE"
    },
    {
        "Category": "Mixed sentiment",
        "Text": "The design is beautiful, but the battery life is disappointing.",
        "Expected": "NEGATIVE"
    },
    {
        "Category": "Sarcasm",
        "Text": "Fantastic! Another three hours wasted waiting for the bus.",
        "Expected": "NEGATIVE"
    },
    {
        "Category": "Neutral or factual",
        "Text": "The package arrived on Tuesday.",
        "Expected": "NEUTRAL"
    },
    {
        "Category": "Short positive",
        "Text": "Loved it!",
        "Expected": "POSITIVE"
    },
    {
        "Category": "Short negative",
        "Text": "Absolutely awful.",
        "Expected": "NEGATIVE"
    },
    {
        "Category": "Ambiguous",
        "Text": "It was something.",
        "Expected": "UNCERTAIN"
    },
    {
        "Category": "Positive with a complaint",
        "Text": "I love the phone, although it gets hot while charging.",
        "Expected": "MIXED"
    }
]

texts = [case["Text"] for case in test_cases]
predictions = sentiment_analyzer(texts)

results_df = pd.DataFrame([
    {
        "Category": case["Category"],
        "Text": case["Text"],
        "Expected": case["Expected"],
        "Predicted": prediction["label"],
        "Confidence": round(prediction["score"], 4)
    }
    for case, prediction in zip(test_cases, predictions)
])

results_df


,Category,Text,Expected,Predicted,Confidence
0,Clearly positive,The food was delicious and the staff were wond...,POSITIVE,POSITIVE,0.9999
1,Clearly negative,The product broke after one day and customer s...,NEGATIVE,NEGATIVE,0.9998
2,Negation,This movie was not good.,NEGATIVE,NEGATIVE,0.9998
3,Mixed sentiment,"The design is beautiful, but the battery life ...",NEGATIVE,NEGATIVE,0.9971
4,Sarcasm,Fantastic! Another three hours wasted waiting ...,NEGATIVE,NEGATIVE,0.9553
5,Neutral or factual,The package arrived on Tuesday.,NEUTRAL,POSITIVE,0.9830
6,Short positive,Loved it!,POSITIVE,POSITIVE,0.9999
7,Short negative,Absolutely awful.,NEGATIVE,NEGATIVE,0.9997
8,Ambiguous,It was something.,UNCERTAIN,POSITIVE,0.9981
9,Positive with a complaint,"I love the phone, although it gets hot while c...",MIXED,POSITIVE,0.9991


## Results and Observations

The pretrained DistilBERT sentiment analysis model was evaluated using 10 manually selected text examples covering positive sentiment, negative sentiment, negation, mixed sentiment, sarcasm, neutral statements, short texts, and ambiguous inputs.

The model produced the expected positive or negative labels for 7 of the 10 examples according to the manually assigned expectations. It handled straightforward positive and negative statements well in this small test set. It also classified the negated sentence and the selected sarcastic example as negative.

However, the model classified a neutral factual statement as positive with high confidence. It also assigned a positive label to an ambiguous statement and a sentence containing both positive and negative aspects. These results demonstrate that a model can produce a confident prediction even when its output does not adequately represent the meaning of the input.

### Key Limitations

1. **Limited label categories:** The selected model supports only positive and negative sentiment labels.
2. **Neutral statements:** Factual statements may be incorrectly forced into a positive or negative category.
3. **Mixed sentiment:** A single label cannot represent both praise and criticism in the same sentence.
4. **Ambiguity and sarcasm:** Interpretation may depend on context that the model does not fully capture.
5. **Confidence interpretation:** Confidence scores should not be treated as proof that a prediction is correct.
6. **Limited evaluation data:** Ten manually selected examples are insufficient to establish general accuracy or performance across different domains.

### Conclusion

The experiment demonstrates that DistilBERT can classify straightforward positive and negative text effectively on the tested examples. However, its binary output categories limit its ability to represent neutral, ambiguous, and mixed sentiment. Future work could evaluate the model on a larger labeled dataset, compare it with a model that supports neutral sentiment, and investigate confidence calibration and domain-specific performance.

# Research Questions and Objectives

## Research Questions

This project investigates the capabilities and limitations of a pretrained DistilBERT model for sentiment analysis.

1. How effectively can DistilBERT classify clearly positive and negative sentences?
2. How does the model handle negation, sarcasm, and mixed sentiment?
3. Can the model correctly interpret neutral and ambiguous statements?
4. Do high confidence scores always correspond to appropriate predictions?
5. What are the limitations and ethical considerations of using automated sentiment analysis?

## Objectives

- Load and use a pretrained DistilBERT sentiment analysis model.
- Test the model using different types of text inputs.
- Examine predicted sentiment labels and confidence scores.
- Identify cases where the model's predictions do not match the intended meaning.
- Discuss the model's limitations, ethical concerns, practical applications, and possible improvements.

## Scope of the Experiment

This experiment uses a pretrained binary sentiment classification model and a small, manually selected set of test sentences. The results are exploratory and are not intended to establish general model accuracy.


## Model Selection and Methodology

### 1. Selected Model

**Model:** DistilBERT for Sentiment Analysis

**Pretrained model identifier:** `distilbert/distilbert-base-uncased-finetuned-sst-2-english`

DistilBERT is a smaller, distilled version of BERT designed to retain much of BERT's language-understanding capability while reducing model size and computational requirements. The selected pretrained model has been fine-tuned on the Stanford Sentiment Treebank (SST-2) dataset for binary sentiment classification.

### 2. Why This Model Was Selected

- It provides a pretrained model that can be used without training a model from scratch.
- It supports text classification through the Hugging Face Transformers pipeline.
- It produces sentiment labels and confidence scores.
- It allows experiments with different types of text, including negation, sarcasm, and mixed sentiment.
- Its binary classification output makes it possible to investigate the limitations of sentiment analysis systems.

### 3. Tools and Technologies

- **Python:** Programming language used for the experiment.
- **PyTorch:** Deep learning framework used to run the model.
- **Hugging Face Transformers:** Library used to load and run the pretrained model.
- **Jupyter Notebook:** Interactive environment for implementing experiments and recording observations.

### 4. Methodology

The experiment follows these steps:

1. Import the required Python libraries.
2. Load the pretrained DistilBERT sentiment analysis pipeline.
3. Prepare sentences representing different sentiment categories.
4. Pass the sentences to the model for prediction.
5. Record the predicted sentiment labels and confidence scores.
6. Compare the predictions with manually assigned expected labels.
7. Analyze successful predictions, unexpected results, and model limitations.

### 5. Evaluation Approach

The model is evaluated through qualitative comparison of its predictions against the expected labels assigned to the selected test sentences. The experiment also examines confidence scores and the model's behavior on challenging inputs.

Because the test set is small and manually selected, the findings are exploratory. They should not be interpreted as a reliable estimate of performance on all sentiment analysis tasks.

## Technical Implementation

### 1. Loading the Model

The Hugging Face Transformers library provides a `pipeline` interface that simplifies text classification. The pretrained DistilBERT model is loaded using the `sentiment-analysis` task.

```python
from transformers import pipeline

MODEL_NAME = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

sentiment_analyzer = pipeline(
    task="sentiment-analysis",
    model=MODEL_NAME,
    device=-1
)

print("DistilBERT sentiment model loaded successfully.")
```

The `device=-1` setting runs inference on the CPU, so a dedicated GPU is not required.

### 2. Generating Predictions

The model accepts a sentence or a list of sentences and returns a predicted label and confidence score for each input.

```python
sentences = [
    "I absolutely love this product.",
    "This is the worst experience I have ever had.",
    "The service was good, but the delivery was disappointing."
]

results = sentiment_analyzer(sentences)

for sentence, result in zip(sentences, results):
    print(f"Text: {sentence}")
    print(f"Predicted sentiment: {result['label']}")
    print(f"Confidence score: {result['score']:.4f}")
    print("-" * 50)
```

### 3. Understanding the Output

- **Text:** The input sentence given to the model.
- **Predicted sentiment:** The model's predicted class, either `POSITIVE` or `NEGATIVE`.
- **Confidence score:** The model's score for its selected class, ranging from 0 to 1.

A high confidence score does not guarantee that a prediction is correct. The model can be highly confident even when it misinterprets neutral, ambiguous, or mixed-sentiment text.

### 4. Batch Processing

The experiment also passes multiple test sentences to the pipeline in one call. Batch processing makes it easier to compare predictions across different types of input while keeping the evaluation procedure consistent.

## Ethical Considerations and Real-World Applications

### 1. Ethical Considerations

**Bias and Fairness:** The model's predictions may reflect biases present in its training data. Its performance may vary across topics, writing styles, and groups of users.

**Misinterpretation of Language:** Sarcasm, cultural expressions, negation, and mixed opinions can be difficult to interpret. Incorrect predictions may misrepresent a person's actual sentiment.

**Privacy:** Text submitted for sentiment analysis may contain personal or confidential information. Applications should minimize unnecessary data collection and protect sensitive user data.

**Overreliance on Confidence Scores:** High confidence scores do not guarantee correct predictions. Decisions should not be based solely on the model's confidence score.

**Human Oversight:** For important decisions, automated sentiment predictions should be reviewed by people rather than treated as definitive judgments about individuals.

### 2. Real-World Applications

- **Customer Feedback Analysis:** Identify positive and negative opinions in product reviews.
- **Product Improvement:** Analyze recurring complaints and customer satisfaction trends.
- **Social Media Monitoring:** Examine public reactions to products, services, or campaigns.
- **Service Quality Evaluation:** Identify potentially negative feedback about customer service.
- **Market Research:** Summarize sentiment patterns to support business analysis.

### 3. Risks in Practical Use

Incorrect sentiment classification can lead to misleading reports, poor business decisions, or unfair conclusions about customers. The model should be tested on representative data from the intended application before deployment.

### 4. Responsible Use

The model should support, rather than replace, human judgment. Developers should evaluate its limitations, protect user privacy, monitor performance, and communicate uncertainty when using sentiment predictions in real-world systems.

## Future Improvements

The following improvements could make the sentiment analysis system more reliable and useful:

1. **Support for Neutral Sentiment:** Evaluate a pretrained model that supports positive, negative, and neutral labels.
2. **Larger Evaluation Dataset:** Test the model on a larger, labeled dataset to obtain more reliable performance measurements.
3. **Mixed-Sentiment Detection:** Explore approaches that can identify positive and negative opinions within the same sentence.
4. **Sarcasm and Context Understanding:** Evaluate specialized models or contextual methods for handling sarcasm and complex language.
5. **Confidence Calibration:** Investigate whether the model's confidence scores correspond to its actual prediction reliability.
6. **Domain-Specific Evaluation:** Test performance on customer reviews, social media posts, and other relevant text domains.
7. **Comparative Analysis:** Compare DistilBERT with other pretrained sentiment analysis models using the same evaluation dataset.

## Conclusion

This project explored sentiment analysis using a pretrained DistilBERT model from the Hugging Face Transformers library. The model was tested on multiple text inputs covering straightforward sentiment, negation, sarcasm, mixed sentiment, neutral statements, and ambiguous language.

The experiments demonstrated that the model handled several straightforward examples correctly but also produced unexpected predictions for neutral and ambiguous inputs. These observations highlight the limitations of binary sentiment classification and demonstrate why confidence scores should be interpreted carefully.

The project provided practical experience in loading a pretrained language model, performing inference, examining predictions, and evaluating model behavior. Further work using larger labeled datasets, models with additional sentiment categories, and more systematic evaluation would help establish its suitability for practical applications.